# 🚀 Jadi Apa — Unattended Render Farm (Kaggle GPU T4 x2 + Blender 5.2.2, v3.0)

**1-File Upload & 1-Click Render Final (`1080×1920`, 24fps, 24 samples + Motion Blur + `post.py`)!**
Cukup upload 1 file **`ep001_kaggle_bundle.zip`** (dari Google Drive `channel-micro-pov/04_episodes/ep001_kecoak_dapur/`) sebagai Kaggle Input Dataset, lalu klik **Save Version → Save & Run All (Commit)**.

Notebook ini otomatis:
1. Mendeteksi & mengekstrak **`ep001_kaggle_bundle.zip`** (berisi **`mas_dimas_ready.blend` v3.0 Semi-Matte**, **seluruh 19 properti Poly Haven CC0 + HDRI + PBR textures + SFX/BGM** agar 100% aman dari rate-limit, `timing_manifest.json`, serta `raw_render/vo/scene_01..08.wav` + `.words.json` — **0 kebutuhan `GEMINI_API_KEY`!**).
2. Mengaktifkan **2× NVIDIA Tesla T4 GPU** paralel (Cycles + OpenImageDenoise + AgX + Motion Blur + `post.py`) dengan **auto-resume per frame (`>10 KB`)**.
3. Menjalankan `compose_final.py` (4-layer audio `-14 LUFS`, `skitter_fast -7.5 dB` bawah VO, subtitle karaoke `faster-whisper` `1 / 1.08` tanpa badge ganda Scene 08) dan menyimpan **`/kaggle/working/final_ep001.mp4`**.
4. Jika terjadi error apapun saat runtime, seluruh `traceback` & 150 baris log terakhir otomatis disimpan ke **`/kaggle/working/error_report.txt`** (bisa langsung dibuka & disalin dari tab Output tanpa screenshot!).

### Pengaturan Kaggle (Panel Kanan — Wajib):
1. **Accelerator:** `GPU T4 x2`
2. **Internet:** `On`
3. **Add Input:** Upload/tambahkan dataset berisi **`ep001_kaggle_bundle.zip`** (baik file `.zip` maupun hasil auto-ekstrak Kaggle).

In [ ]:
# ---- PENGATURAN JOB RENDER (DEFAULT SIAP RENDER FINAL EP001!) ----
REPO_URL       = "https://github.com/ruangcermat/channel-micro-pov.git"
REPO_BRANCH    = "main"
EPISODE_SPEC   = "04_episodes/ep001_kecoak_dapur/ep001_spec.json"
RENDER_MODE    = "final"      # "storyboard" | "motion_check" (360x640) | "preview" (540x960) | "final" (1080x1920 24fps)
RENDER_ENGINE  = "CYCLES"     # "CYCLES" (2xT4 GPU + OIDN) | "EEVEE"
SINGLE_SCENE   = 0            # 0 = render semua 8 scene; 1..8 = render 1 scene saja
TTS_VOICE      = "Puck"
VO_SPEED       = 1.08
SKIP_EXISTING_VO = True

# Opsional fallback jika tidak upload ep001_kaggle_bundle.zip ke Kaggle Input:
BUNDLE_GDRIVE_ID          = ""
MAS_DIMAS_BLEND_GDRIVE_ID = ""
FAMILY_BLEND_GDRIVE_ID    = ""
ZIP_FRAMES_OUTPUT = False     # True jika ingin ikut menyimpan frames.zip di Output tab

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

In [ ]:
# ---- 1. AUTO ERROR LOGGER (error_report.txt), SECRETS, DEPENDENSI & BLENDER 5.2.2 LTS ----
import datetime
import glob
import json
import os
import shutil
import subprocess
import traceback
import zipfile
from collections import deque
from pathlib import Path

ROOT_WORK = Path("/kaggle/working")
REPO_DIR = ROOT_WORK / "channel-micro-pov"
ASSETS_DIR = ROOT_WORK / "assets_cache"
EPISODE_SLUG = Path(EPISODE_SPEC).parent.name
WORK_DIR = ROOT_WORK / "ep_work" / EPISODE_SLUG
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
(ASSETS_DIR / "models_3d").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "raw_render" / "vo").mkdir(parents=True, exist_ok=True)

ERROR_REPORT_PATH = ROOT_WORK / "error_report.txt"
_LAST_LOG_TAIL = deque(maxlen=150)

def save_error_report(step_name: str, exc: BaseException):
    """Simpan traceback lengkap & 150 baris output terakhir ke /kaggle/working/error_report.txt."""
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    tb_text = traceback.format_exc()
    tail_text = "".join(_LAST_LOG_TAIL)
    report = (
        f"=== JADI APA PIPELINE ERROR REPORT ===\n"
        f"Timestamp     : {ts}\n"
        f"Environment   : Kaggle Notebook\n"
        f"Step          : {step_name}\n"
        f"Episode Spec  : {EPISODE_SPEC}\n"
        f"Render Mode   : {RENDER_MODE} | Engine: {RENDER_ENGINE} | Scene: {SINGLE_SCENE or 'ALL'}\n"
        f"Work Dir      : {WORK_DIR}\n"
        f"Exception     : {type(exc).__name__}: {exc}\n\n"
        f"--- FULL TRACEBACK ---\n{tb_text}\n"
        f"--- LAST 150 LINES OF OUTPUT ---\n{tail_text}\n"
    )
    for dest in (ERROR_REPORT_PATH, WORK_DIR / "error_report.txt"):
        try:
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_text(report, encoding="utf-8")
        except Exception as write_err:
            print(f"⚠️ Gagal menulis {dest}: {write_err}")
    print("\n" + "=" * 72)
    print(f"❌ ERROR PADA LANGKAH: {step_name}")
    print(f"📄 Traceback lengkap otomatis disimpan ke: {ERROR_REPORT_PATH}")
    print("   (Buka tab Output Kaggle -> unduh/copy error_report.txt tanpa screenshot!)")
    print("=" * 72)
    print(tb_text)

def run_step(cmd, step_name: str = ""):
    """Jalankan perintah dengan output real-time + otomatis tulis error_report.txt bila gagal."""
    label = step_name or (" ".join(str(x) for x in cmd[:3]))
    _LAST_LOG_TAIL.clear()
    try:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            _LAST_LOG_TAIL.append(line)
            print(line, end="", flush=True)
        rc = proc.wait()
        if rc != 0:
            raise subprocess.CalledProcessError(rc, cmd)
    except Exception as exc:
        save_error_report(label, exc)
        raise

def run_guarded(step_name: str, fn):
    try:
        return fn()
    except Exception as exc:
        save_error_report(step_name, exc)
        raise

def _step_setup():
    github_token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for key in ("GEMINI_API_KEY", "GITHUB_TOKEN"):
            try:
                val = secrets.get_secret(key)
                if val and val.strip():
                    if key == "GITHUB_TOKEN":
                        github_token = val.strip()
                    else:
                        os.environ[key] = val.strip()
                    print(f"✅ Loaded Kaggle Secret: {key}")
            except Exception:
                pass
    except Exception:
        pass

    auth_repo_url = REPO_URL
    if github_token and REPO_URL.startswith("https://"):
        auth_repo_url = REPO_URL.replace("https://", f"https://{github_token}@", 1)

    if (REPO_DIR / ".git").exists():
        run_step(["git", "-C", str(REPO_DIR), "fetch", "origin"], "git fetch")
        run_step(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], "git checkout")
        run_step(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], "git pull")
    else:
        run_step(["git", "clone", "-b", REPO_BRANCH, auth_repo_url, str(REPO_DIR)], "git clone")

    run_step(
        ["bash", "-lc", "apt-get -qq update > /dev/null && apt-get -qq install -y ffmpeg fonts-dejavu-core libxrender1 libxxf86vm1 libxfixes3 libxi6 libxkbcommon0 libsm6 libgl1 libegl1 libglu1-mesa > /dev/null"],
        "apt-get install",
    )
    run_step(["pip", "install", "-q", "--upgrade", "-r", str(REPO_DIR / "03_pipeline/requirements.txt"), "gdown"], "pip install")

    blender_dir = ROOT_WORK / "blender52"
    blender_bin = blender_dir / "blender"
    if not blender_bin.exists():
        tar_path = "/tmp/blender-5.2.2-linux-x64.tar.xz"
        run_step(["wget", "-q", "https://download.blender.org/release/Blender5.2/blender-5.2.2-linux-x64.tar.xz", "-O", tar_path], "wget Blender 5.2.2")
        blender_dir.mkdir(parents=True, exist_ok=True)
        run_step(["tar", "-xf", tar_path, "-C", str(blender_dir), "--strip-components=1"], "extract Blender")
        os.remove(tar_path)
    run_step([str(blender_bin), "--version"], "blender --version")
    return blender_bin

BLENDER_BIN = run_guarded("Cell 1: Setup Environment & Blender 5.2.2", _step_setup)

In [ ]:
# ---- 2. AUTO-EXTRACT 1-FILE BUNDLE (ep001_kaggle_bundle.zip) + FULL 02_assets CACHE + CACHED VO ----
def _step_unpack_bundle():
    mas_dimas_cache = ASSETS_DIR / "models_3d" / "mas_dimas_ready.blend"
    family_cache = ASSETS_DIR / "models_3d" / "family_ready.blend"
    vo_dest = WORK_DIR / "raw_render" / "vo"
    vo_dest.mkdir(parents=True, exist_ok=True)

    # 2a. Auto-detect & extract *_kaggle_bundle.zip (atau unduh via BUNDLE_GDRIVE_ID)
    bundle_zips = sorted(glob.glob("/kaggle/input/**/*kaggle_bundle*.zip", recursive=True))
    if not bundle_zips and BUNDLE_GDRIVE_ID:
        dl_zip = ROOT_WORK / "kaggle_bundle.zip"
        run_step(["gdown", "--id", BUNDLE_GDRIVE_ID, "-O", str(dl_zip)], "gdown kaggle_bundle.zip")
        if dl_zip.exists():
            bundle_zips = [str(dl_zip)]

    search_roots = ["/kaggle/input"]
    if bundle_zips:
        extract_dir = ROOT_WORK / "_bundle_extracted"
        extract_dir.mkdir(parents=True, exist_ok=True)
        for zp in bundle_zips:
            print(f"📦 Mengekstrak 1-File Kaggle Bundle: {zp}")
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(extract_dir)
        search_roots.insert(0, str(extract_dir))

    # 2b. Salin seluruh pohon cache `02_assets/` dari bundle (models_3d, textures, hdri, sfx, bgm, asset_catalog.json)
    #     agar 100% aman dari rate-limit Poly Haven / jaringan Kaggle!
    copied_assets = 0
    for root in search_roots:
        for assets_src in sorted(glob.glob(f"{root}/**/02_assets", recursive=True)):
            src_root = Path(assets_src)
            if not src_root.is_dir():
                continue
            for item in src_root.rglob("*"):
                if item.is_file():
                    rel = item.relative_to(src_root)
                    dst = ASSETS_DIR / rel
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(item, dst)
                        copied_assets += 1
    if copied_assets:
        print(f"✅ Disalin {copied_assets} file cache lokal 02_assets/ (Poly Haven GLTF + HDRI + PBR + SFX/BGM + Mas Dimas)!")

    def _find_all(pattern: str):
        hits = []
        for root in search_roots:
            hits.extend(sorted(glob.glob(f"{root}/**/{pattern}", recursive=True)))
        return hits

    # 2c. Master Mas Dimas (mas_dimas_ready.blend)
    if not mas_dimas_cache.exists():
        found_md = _find_all("mas_dimas_ready.blend")
        if found_md:
            shutil.copy2(found_md[0], mas_dimas_cache)
            print(f"✅ Copied mas_dimas_ready.blend -> {mas_dimas_cache}")
        elif MAS_DIMAS_BLEND_GDRIVE_ID:
            run_step(["gdown", "--id", MAS_DIMAS_BLEND_GDRIVE_ID, "-O", str(mas_dimas_cache)], "gdown mas_dimas_ready.blend")
    if mas_dimas_cache.exists():
        os.environ["MICRO_POV_MAS_DIMAS_BLEND"] = str(mas_dimas_cache)
        print(f"✅ Master Mas Dimas v3.0 siap ({mas_dimas_cache.stat().st_size / 1e6:.1f} MB)")

    # 2d. Fallback family_ready.blend (jika belum ada mas_dimas_ready.blend)
    if not family_cache.exists():
        found_fb = _find_all("family_ready.blend")
        if found_fb:
            shutil.copy2(found_fb[0], family_cache)
        elif FAMILY_BLEND_GDRIVE_ID:
            run_step(["gdown", "--id", FAMILY_BLEND_GDRIVE_ID, "-O", str(family_cache)], "gdown family_ready.blend")
    if family_cache.exists():
        os.environ["MICRO_POV_FAMILY_BLEND"] = str(family_cache)

    assert mas_dimas_cache.exists() or family_cache.exists(), (
        "mas_dimas_ready.blend tidak ditemukan! Tambahkan dataset berisi ep001_kaggle_bundle.zip "
        "di panel Add Input Kaggle lalu jalankan ulang."
    )

    # 2e. Copy cached VO WAVs + faster-whisper .words.json caches + timing_manifest.json
    copied_wav = 0
    for wav in _find_all("scene_*.wav"):
        dst = vo_dest / os.path.basename(wav)
        if not dst.exists():
            shutil.copy2(wav, dst)
            copied_wav += 1

    copied_words = 0
    for wj in _find_all("scene_*.words.json"):
        dst = vo_dest / os.path.basename(wj)
        if not dst.exists():
            shutil.copy2(wj, dst)
            copied_words += 1

    found_manifest = _find_all("timing_manifest.json")
    if found_manifest and not (WORK_DIR / "timing_manifest.json").exists():
        shutil.copy2(found_manifest[0], WORK_DIR / "timing_manifest.json")
        print(f"✅ Copied timing_manifest.json -> {WORK_DIR / 'timing_manifest.json'}")

    print(f"✅ Cache VO siap: {copied_wav} WAV + {copied_words} faster-whisper .words.json (SKIP_EXISTING_VO={SKIP_EXISTING_VO})")

run_guarded("Cell 2: Unpack Kaggle Bundle & Assets", _step_unpack_bundle)

In [ ]:
# ---- 3. FETCH POLY HAVEN CC0 PROPS + VERIFIKASI TIMING AUDIO-FIRST + SFX/BGM ----
SPEC_PATH = REPO_DIR / EPISODE_SPEC

def _step_prep_assets_and_audio():
    run_step([
        "python3", str(REPO_DIR / "03_pipeline/scripts/fetch_assets.py"),
        "--spec", str(SPEC_PATH),
        "--assets-dir", str(ASSETS_DIR),
    ], "Cell 3a: fetch_assets.py")

    tts_cmd = [
        "python3", str(REPO_DIR / "03_pipeline/scripts/generate_tts.py"),
        "--spec", str(SPEC_PATH),
        "--work-dir", str(WORK_DIR),
        "--voice", TTS_VOICE,
        "--fps", "24",
        "--pad-sec", "0.06",
        "--vo-speed", str(VO_SPEED),
    ]
    if SKIP_EXISTING_VO:
        tts_cmd.append("--skip-existing")
    run_step(tts_cmd, "Cell 3b: generate_tts.py (cached VO)")

    run_step([
        "python3", str(REPO_DIR / "03_pipeline/scripts/prepare_audio.py"),
        "--spec", str(SPEC_PATH),
        "--assets-dir", str(ASSETS_DIR),
    ], "Cell 3c: prepare_audio.py")

run_guarded("Cell 3: Fetch Props & Prepare Audio", _step_prep_assets_and_audio)

In [ ]:
# ---- 4. RENDER 3D BLENDER 5.2.2 (2xT4 GPU + AUTO-RESUME >10KB + post.py) ----
def _step_blender_render():
    render_cmd = [
        str(BLENDER_BIN),
        "--background",
        "--python", str(REPO_DIR / "03_pipeline/scripts/build_and_render.py"),
        "--",
        "--spec", str(SPEC_PATH),
        "--timing", str(WORK_DIR / "timing_manifest.json"),
        "--assets-dir", str(ASSETS_DIR),
        "--work-dir", str(WORK_DIR),
        "--mode", RENDER_MODE,
        "--engine", RENDER_ENGINE,
    ]
    if SINGLE_SCENE > 0:
        render_cmd.extend(["--scene", str(SINGLE_SCENE)])
    run_step(render_cmd, f"Cell 4: Blender 3D Render ({RENDER_MODE})")

run_guarded("Cell 4: Blender 3D Render", _step_blender_render)

In [ ]:
# ---- 5. SUBTITLE faster-whisper .ASS 9:16 + 4-LAYER AUDIO (-14 LUFS) + EXPORT /kaggle/working/ ----
def _step_compose_and_export():
    run_step([
        "python3", str(REPO_DIR / "03_pipeline/scripts/compose_final.py"),
        "--spec", str(SPEC_PATH),
        "--timing", str(WORK_DIR / "timing_manifest.json"),
        "--assets-dir", str(ASSETS_DIR),
        "--work-dir", str(WORK_DIR),
        "--mode", RENDER_MODE,
    ], "Cell 5: compose_final.py")

    manifest = json.loads((WORK_DIR / "timing_manifest.json").read_text(encoding="utf-8"))
    ep_id = manifest["episode_id"].lower()
    if RENDER_MODE == "final":
        out_name = f"final_{ep_id}.mp4"
    elif RENDER_MODE == "storyboard":
        out_name = f"animatic_{ep_id}.mp4"
    elif RENDER_MODE == "motion_check":
        out_name = f"motion_check_{ep_id}.mp4"
    else:
        out_name = f"preview_{ep_id}.mp4"

    src_mp4 = WORK_DIR / out_name
    dst_mp4 = ROOT_WORK / out_name
    shutil.copy2(src_mp4, dst_mp4)
    print(f"🎉 OUTPUT FINAL SIAP UNDUH: {dst_mp4} ({dst_mp4.stat().st_size / 1e6:.2f} MB)")

    for png_name in ("storyboard_sheet.png", "storyboard_grid.png"):
        src_p = WORK_DIR / png_name
        if src_p.exists():
            shutil.copy2(src_p, ROOT_WORK / png_name)

    if ZIP_FRAMES_OUTPUT:
        scenes_dir = WORK_DIR / "raw_render" / f"scenes_{RENDER_MODE}"
        if scenes_dir.exists():
            run_step(["zip", "-qr", str(ROOT_WORK / f"frames_{ep_id}.zip"), str(scenes_dir)], "zip frames")

run_guarded("Cell 5: Compose Final MP4 & Export", _step_compose_and_export)

In [ ]:
# ---- 6. BERSIHKAN CACHE SEMENTARA (SIMPAN MP4 & error_report.txt DI TAB OUTPUT) ----
!rm -rf /kaggle/working/blender52 /kaggle/working/assets_cache /kaggle/working/channel-micro-pov /kaggle/working/ep_work /kaggle/working/_bundle_extracted
!ls -lh /kaggle/working